# Tiny intent model — real CPU training
A small linear softmax classifier (87 parameters), not an LLM. Synthetic data only. No network/download/GPU. Scores are not calibrated; six validation/test examples do not prove general quality.
Review and authorize any Kaggle execution separately. Never place credentials in this notebook.


In [ ]:
import base64, json
JOB = json.loads(base64.b64decode("eyJjb25maWciOnsiYmF0Y2hfc2l6ZSI6NCwiZXBvY2hzIjoyMCwibGVhcm5pbmdfcmF0ZSI6MC44LCJzZWVkIjo3fSwiZGF0YXNldCI6eyJpZCI6InN5bnRoZXRpYy1pbnRlbnRzLTMwLXYxIiwic2hhMjU2IjoiODUyNGVmNDZkOWJhYTBlNmQzMDMwYzMxZDExMmY3Y2U0MmYwNzM4ZGRhYzhlZDdlOWFlYzIxNGY5MGJmNTE3OCIsInRlc3RfZXhhbXBsZXMiOjYsInRyYWluX2V4YW1wbGVzIjoxOCwidmFsaWRhdGlvbl9leGFtcGxlcyI6Nn0sImVudmlyb25tZW50Ijp7ImhhcmR3YXJlIjp7ImRldmljZSI6ImNwdSIsIm1lbW9yeV9tYiI6NTEyfSwiaW50ZXJuZXQiOmZhbHNlLCJsaWJyYXJpZXMiOnsic3RkbGliIjoiYnVpbHRpbiJ9LCJweXRob25fbWluIjoiMy4xMCJ9LCJleHBlcmltZW50X3R5cGUiOiJ0cmFpbmluZyIsImlkIjoiaW50ZW50LXRyYWluaW5nLXN0dWR5IiwibW9kZWwiOnsiaWQiOiJ0aW55LWludGVudC1zb2Z0bWF4LXYxIiwicGFyYW1ldGVycyI6ODcsIndlaWdodHNfc2hhMjU2IjpudWxsfSwib2JqZWN0aXZlIjoiUnVuIHJlYWwgdGlueSBsaW5lYXItbW9kZWwgdHJhaW5pbmcgb24gc3ludGhldGljIGludGVudCBkYXRhOyBubyBMTE0gb3IgZXh0ZXJuYWwgZG93bmxvYWQuIiwicHJvdmVuYW5jZSI6eyJjb3JlX3NoYTI1NiI6Ijk3NTE0OTU0ZGE3MjEzMWVjNTgwOTExNDA4MDlkODkzNDk5ZmEzMTEyM2NkMmY1ZmMxYjZjYmE1MTdjYzExZTkiLCJwcm9kdWNlciI6InB5dGhvbi1haS10cmFpbmVyIiwidGVtcGxhdGVfdmVyc2lvbiI6MX0sInJlY2lwZV9pZCI6ImludGVudC10cmFpbmluZyIsInJlc291cmNlcyI6eyJtYXhfYXJ0aWZhY3RfYnl0ZXMiOjEwNDg1NzYsInRpbWVvdXRfc2Vjb25kcyI6MzAwfSwic2NoZW1hX3ZlcnNpb24iOjJ9"))


In [ ]:
"""Tiny real softmax classifier using only the Python standard library.

This is a linear text model, not an LLM. Scores are not calibrated confidence.
Training data and checkpoints are public educational fixtures, never pickle.
"""
import copy
import hashlib
import json
import math
import random
import re

LABELS = ['billing', 'support', 'schedule']
TOKENIZER = 'lower-ascii-words-v1'

def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()

def tokenize(text):
    if not isinstance(text, str) or len(text) > 512:
        raise ValueError('Input must be a string with at most 512 characters')
    return re.findall(r'[a-z]+', text.lower())

def validate_dataset(data):
    if set(data) != {'version', 'id', 'examples'} or data['version'] != 1:
        raise ValueError('Invalid dataset schema')
    rows=data['examples']
    if not isinstance(rows,list) or not 9 <= len(rows) <= 2048:
        raise ValueError('Dataset size outside the educational budget')
    ids=set(); texts=set()
    for row in rows:
        if set(row)!={'id','text','label','split'} or row['label'] not in LABELS or row['split'] not in ['train','validation','test']:
            raise ValueError('Invalid dataset row')
        if row['id'] in ids or row['text'].lower() in texts:
            raise ValueError('Duplicate id/text leaks examples across splits')
        ids.add(row['id']);texts.add(row['text'].lower());tokenize(row['text'])
    for split in ['train','validation','test']:
        if {row['label'] for row in rows if row['split']==split}!=set(LABELS):
            raise ValueError('Every split must contain all classes')
    return data

def initialize(rows, seed):
    vocab=sorted({word for row in rows for word in tokenize(row['text'])})
    if not vocab or len(vocab)>4096:raise ValueError('Invalid vocabulary size')
    rng=random.Random(seed)
    return {'version':1,'kind':'tiny-intent-softmax','tokenizer':TOKENIZER,'labels':LABELS[:], 'vocab':vocab,
            'weights':[[rng.uniform(-.01,.01) for _ in vocab] for _ in LABELS], 'bias':[0.0]*len(LABELS)}

def validate_model(model):
    if set(model)!={'version','kind','tokenizer','labels','vocab','weights','bias'} or model['version']!=1 or model['kind']!='tiny-intent-softmax' or model['tokenizer']!=TOKENIZER or model['labels']!=LABELS:
        raise ValueError('Invalid model schema')
    vocab=model['vocab']
    if not isinstance(vocab,list) or not 1<=len(vocab)<=4096 or vocab!=sorted(set(vocab)) or not all(isinstance(w,str) and re.fullmatch('[a-z]+',w) for w in vocab):
        raise ValueError('Invalid vocabulary')
    if len(model['weights'])!=len(LABELS) or len(model['bias'])!=len(LABELS) or any(len(row)!=len(vocab) for row in model['weights']):
        raise ValueError('Incompatible model dimensions')
    values=model['bias']+[v for row in model['weights'] for v in row]
    if not all(type(v) in [int,float] and math.isfinite(v) and abs(v)<1e6 for v in values):
        raise ValueError('Nonfinite/unbounded model parameter')
    return model

def features(model, text):
    indexes={word:i for i,word in enumerate(model['vocab'])}; result={}
    for word in tokenize(text):
        if word in indexes:result[indexes[word]]=result.get(indexes[word],0)+1
    return result

def probabilities(model, text):
    values=features(model,text)
    logits=[b+sum(model['weights'][k][j]*count for j,count in values.items()) for k,b in enumerate(model['bias'])]
    peak=max(logits);exps=[math.exp(v-peak) for v in logits];total=sum(exps)
    return [v/total for v in exps]

def predict(model,text):
    scores=probabilities(model,text);index=max(range(len(scores)),key=scores.__getitem__)
    return {'label':model['labels'][index], 'scores':dict(zip(model['labels'],scores)), 'word_tokens':len(tokenize(text))}

def evaluate(model, rows):
    if not rows:raise ValueError('Cannot evaluate an empty dataset')
    correct=0;loss=0
    for row in rows:
        values=probabilities(model,row['text']);target=model['labels'].index(row['label'])
        correct+=max(range(len(values)),key=values.__getitem__)==target;loss-=math.log(max(values[target],1e-15))
    return {'accuracy':correct/len(rows),'loss':loss/len(rows),'correct':correct,'examples':len(rows)}

def train(data, config, event=lambda row:None):
    validate_dataset(data)
    if set(config)!={'seed','epochs','batch_size','learning_rate'} or type(config['seed']) is not int or not 0<=config['seed']<2**31 or type(config['epochs']) is not int or not 1<=config['epochs']<=100 or type(config['batch_size']) is not int or not 1<=config['batch_size']<=16 or type(config['learning_rate']) not in [int,float] or not math.isfinite(config['learning_rate']) or not .001<=config['learning_rate']<=1:
        raise ValueError('Invalid bounded training configuration')
    rows={split:[r for r in data['examples'] if r['split']==split] for split in ['train','validation','test']}
    model=initialize(rows['train'],config['seed']);rng=random.Random(config['seed'])
    before=evaluate(model,rows['validation']);history=[];best=None;best_loss=math.inf;best_epoch=None
    for epoch in range(1,config['epochs']+1):
        order=rows['train'][:];rng.shuffle(order)
        for start in range(0,len(order),config['batch_size']):
            batch=order[start:start+config['batch_size']];gw=[[0.0]*len(model['vocab']) for _ in LABELS];gb=[0.0]*len(LABELS)
            for row in batch:
                x=features(model,row['text']);values=probabilities(model,row['text']);target=LABELS.index(row['label'])
                for k in range(len(LABELS)):
                    error=values[k]-(k==target);gb[k]+=error
                    for j,count in x.items():gw[k][j]+=error*count
            rate=config['learning_rate']/len(batch)
            for k in range(len(LABELS)):
                model['bias'][k]-=rate*gb[k]
                for j in range(len(model['vocab'])):model['weights'][k][j]-=rate*gw[k][j]
        training=evaluate(model,rows['train']);validation=evaluate(model,rows['validation'])
        row={'event':'epoch','epoch':epoch,'training_loss':training['loss'],'validation_loss':validation['loss'],'validation_accuracy':validation['accuracy']};history.append(row);event(row)
        if validation['loss']<best_loss:best=copy.deepcopy(model);best_loss=validation['loss'];best_epoch=epoch
    # Test examples are not used to build vocabulary or select the checkpoint.
    return {'model':validate_model(best),'history':history,'selected_epoch':best_epoch,'before_validation':before,
            'after_validation':evaluate(best,rows['validation']),'heldout_test':evaluate(best,rows['test']),
            'splits':{split:[row['id'] for row in rows[split]] for split in rows}}


In [ ]:
DATA_BYTES = base64.b64decode("ewogICJ2ZXJzaW9uIjogMSwKICAiaWQiOiAic3ludGhldGljLWludGVudHMtMzAtdjEiLAogICJleGFtcGxlcyI6IFsKICAgIHsKICAgICAgImlkIjogImJpbGxpbmctdHJhaW4tMSIsCiAgICAgICJ0ZXh0IjogImludm9pY2UgcGF5bWVudCBmYWlsZWQiLAogICAgICAibGFiZWwiOiAiYmlsbGluZyIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJiaWxsaW5nLXRyYWluLTIiLAogICAgICAidGV4dCI6ICJyZWZ1bmQgaW52b2ljZSBhbW91bnQiLAogICAgICAibGFiZWwiOiAiYmlsbGluZyIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJiaWxsaW5nLXRyYWluLTMiLAogICAgICAidGV4dCI6ICJiaWxsaW5nIHBheW1lbnQgcmVjZWlwdCIsCiAgICAgICJsYWJlbCI6ICJiaWxsaW5nIiwKICAgICAgInNwbGl0IjogInRyYWluIgogICAgfSwKICAgIHsKICAgICAgImlkIjogImJpbGxpbmctdHJhaW4tNCIsCiAgICAgICJ0ZXh0IjogImludm9pY2UgcmVmdW5kIHJlcXVlc3QiLAogICAgICAibGFiZWwiOiAiYmlsbGluZyIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJiaWxsaW5nLXRyYWluLTUiLAogICAgICAidGV4dCI6ICJiaWxsaW5nIGNoYXJnZSBhbW91bnQiLAogICAgICAibGFiZWwiOiAiYmlsbGluZyIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJiaWxsaW5nLXRyYWluLTYiLAogICAgICAidGV4dCI6ICJwYXltZW50IGNoYXJnZSByZWNlaXB0IiwKICAgICAgImxhYmVsIjogImJpbGxpbmciLAogICAgICAic3BsaXQiOiAidHJhaW4iCiAgICB9LAogICAgewogICAgICAiaWQiOiAiYmlsbGluZy12YWxpZGF0aW9uLTEiLAogICAgICAidGV4dCI6ICJpbnZvaWNlIHBheW1lbnQgcmVxdWVzdCIsCiAgICAgICJsYWJlbCI6ICJiaWxsaW5nIiwKICAgICAgInNwbGl0IjogInZhbGlkYXRpb24iCiAgICB9LAogICAgewogICAgICAiaWQiOiAiYmlsbGluZy12YWxpZGF0aW9uLTIiLAogICAgICAidGV4dCI6ICJyZWZ1bmQgYmlsbGluZyByZWNlaXB0IiwKICAgICAgImxhYmVsIjogImJpbGxpbmciLAogICAgICAic3BsaXQiOiAidmFsaWRhdGlvbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJiaWxsaW5nLXRlc3QtMSIsCiAgICAgICJ0ZXh0IjogImJpbGxpbmcgaW52b2ljZSBhbW91bnQiLAogICAgICAibGFiZWwiOiAiYmlsbGluZyIsCiAgICAgICJzcGxpdCI6ICJ0ZXN0IgogICAgfSwKICAgIHsKICAgICAgImlkIjogImJpbGxpbmctdGVzdC0yIiwKICAgICAgInRleHQiOiAicmVmdW5kIHBheW1lbnQgY2hhcmdlIiwKICAgICAgImxhYmVsIjogImJpbGxpbmciLAogICAgICAic3BsaXQiOiAidGVzdCIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzdXBwb3J0LXRyYWluLTEiLAogICAgICAidGV4dCI6ICJzb2Z0d2FyZSBlcnJvciByZXBvcnQiLAogICAgICAibGFiZWwiOiAic3VwcG9ydCIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzdXBwb3J0LXRyYWluLTIiLAogICAgICAidGV4dCI6ICJzdXBwb3J0IGxvZ2luIGlzc3VlIiwKICAgICAgImxhYmVsIjogInN1cHBvcnQiLAogICAgICAic3BsaXQiOiAidHJhaW4iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic3VwcG9ydC10cmFpbi0zIiwKICAgICAgInRleHQiOiAicmVzZXQgYWNjb3VudCBwYXNzd29yZCIsCiAgICAgICJsYWJlbCI6ICJzdXBwb3J0IiwKICAgICAgInNwbGl0IjogInRyYWluIgogICAgfSwKICAgIHsKICAgICAgImlkIjogInN1cHBvcnQtdHJhaW4tNCIsCiAgICAgICJ0ZXh0IjogImxvZ2luIGVycm9yIGhlbHAiLAogICAgICAibGFiZWwiOiAic3VwcG9ydCIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzdXBwb3J0LXRyYWluLTUiLAogICAgICAidGV4dCI6ICJzdXBwb3J0IHNvZnR3YXJlIGlzc3VlIiwKICAgICAgImxhYmVsIjogInN1cHBvcnQiLAogICAgICAic3BsaXQiOiAidHJhaW4iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic3VwcG9ydC10cmFpbi02IiwKICAgICAgInRleHQiOiAicGFzc3dvcmQgcmVzZXQgaGVscCIsCiAgICAgICJsYWJlbCI6ICJzdXBwb3J0IiwKICAgICAgInNwbGl0IjogInRyYWluIgogICAgfSwKICAgIHsKICAgICAgImlkIjogInN1cHBvcnQtdmFsaWRhdGlvbi0xIiwKICAgICAgInRleHQiOiAic3VwcG9ydCBlcnJvciBpc3N1ZSIsCiAgICAgICJsYWJlbCI6ICJzdXBwb3J0IiwKICAgICAgInNwbGl0IjogInZhbGlkYXRpb24iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic3VwcG9ydC12YWxpZGF0aW9uLTIiLAogICAgICAidGV4dCI6ICJsb2dpbiBwYXNzd29yZCByZXNldCIsCiAgICAgICJsYWJlbCI6ICJzdXBwb3J0IiwKICAgICAgInNwbGl0IjogInZhbGlkYXRpb24iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic3VwcG9ydC10ZXN0LTEiLAogICAgICAidGV4dCI6ICJzb2Z0d2FyZSBzdXBwb3J0IGhlbHAiLAogICAgICAibGFiZWwiOiAic3VwcG9ydCIsCiAgICAgICJzcGxpdCI6ICJ0ZXN0IgogICAgfSwKICAgIHsKICAgICAgImlkIjogInN1cHBvcnQtdGVzdC0yIiwKICAgICAgInRleHQiOiAiYWNjb3VudCBsb2dpbiBlcnJvciIsCiAgICAgICJsYWJlbCI6ICJzdXBwb3J0IiwKICAgICAgInNwbGl0IjogInRlc3QiCiAgICB9LAogICAgewogICAgICAiaWQiOiAic2NoZWR1bGUtdHJhaW4tMSIsCiAgICAgICJ0ZXh0IjogInNjaGVkdWxlIG1lZXRpbmcgdGltZSIsCiAgICAgICJsYWJlbCI6ICJzY2hlZHVsZSIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzY2hlZHVsZS10cmFpbi0yIiwKICAgICAgInRleHQiOiAiYm9vayBjYWxlbmRhciBhcHBvaW50bWVudCIsCiAgICAgICJsYWJlbCI6ICJzY2hlZHVsZSIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzY2hlZHVsZS10cmFpbi0zIiwKICAgICAgInRleHQiOiAicmVzY2hlZHVsZSBtZWV0aW5nIHRvbW9ycm93IiwKICAgICAgImxhYmVsIjogInNjaGVkdWxlIiwKICAgICAgInNwbGl0IjogInRyYWluIgogICAgfSwKICAgIHsKICAgICAgImlkIjogInNjaGVkdWxlLXRyYWluLTQiLAogICAgICAidGV4dCI6ICJjYWxlbmRhciBtZWV0aW5nIGludml0YXRpb24iLAogICAgICAibGFiZWwiOiAic2NoZWR1bGUiLAogICAgICAic3BsaXQiOiAidHJhaW4iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic2NoZWR1bGUtdHJhaW4tNSIsCiAgICAgICJ0ZXh0IjogImFwcG9pbnRtZW50IHNjaGVkdWxlIHRvbW9ycm93IiwKICAgICAgImxhYmVsIjogInNjaGVkdWxlIiwKICAgICAgInNwbGl0IjogInRyYWluIgogICAgfSwKICAgIHsKICAgICAgImlkIjogInNjaGVkdWxlLXRyYWluLTYiLAogICAgICAidGV4dCI6ICJib29rIHRpbWUgaW52aXRhdGlvbiIsCiAgICAgICJsYWJlbCI6ICJzY2hlZHVsZSIsCiAgICAgICJzcGxpdCI6ICJ0cmFpbiIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzY2hlZHVsZS12YWxpZGF0aW9uLTEiLAogICAgICAidGV4dCI6ICJzY2hlZHVsZSBjYWxlbmRhciBtZWV0aW5nIiwKICAgICAgImxhYmVsIjogInNjaGVkdWxlIiwKICAgICAgInNwbGl0IjogInZhbGlkYXRpb24iCiAgICB9LAogICAgewogICAgICAiaWQiOiAic2NoZWR1bGUtdmFsaWRhdGlvbi0yIiwKICAgICAgInRleHQiOiAiYm9vayBhcHBvaW50bWVudCB0b21vcnJvdyIsCiAgICAgICJsYWJlbCI6ICJzY2hlZHVsZSIsCiAgICAgICJzcGxpdCI6ICJ2YWxpZGF0aW9uIgogICAgfSwKICAgIHsKICAgICAgImlkIjogInNjaGVkdWxlLXRlc3QtMSIsCiAgICAgICJ0ZXh0IjogInJlc2NoZWR1bGUgY2FsZW5kYXIgYXBwb2ludG1lbnQiLAogICAgICAibGFiZWwiOiAic2NoZWR1bGUiLAogICAgICAic3BsaXQiOiAidGVzdCIKICAgIH0sCiAgICB7CiAgICAgICJpZCI6ICJzY2hlZHVsZS10ZXN0LTIiLAogICAgICAidGV4dCI6ICJtZWV0aW5nIHRpbWUgaW52aXRhdGlvbiIsCiAgICAgICJsYWJlbCI6ICJzY2hlZHVsZSIsCiAgICAgICJzcGxpdCI6ICJ0ZXN0IgogICAgfQogIF0KfQo=")
assert hashlib.sha256(DATA_BYTES).hexdigest() == "8524ef46d9baa0e6d3030c31d112f7ce42f0738ddac8ed7e9aec214f90bf5178"
DATA = json.loads(DATA_BYTES)
REFERENCE_MODEL_BYTES = base64.b64decode("ewogICJ2ZXJzaW9uIjogMSwKICAia2luZCI6ICJ0aW55LWludGVudC1zb2Z0bWF4IiwKICAidG9rZW5pemVyIjogImxvd2VyLWFzY2lpLXdvcmRzLXYxIiwKICAibGFiZWxzIjogWwogICAgImJpbGxpbmciLAogICAgInN1cHBvcnQiLAogICAgInNjaGVkdWxlIgogIF0sCiAgInZvY2FiIjogWwogICAgImFjY291bnQiLAogICAgImFtb3VudCIsCiAgICAiYXBwb2ludG1lbnQiLAogICAgImJpbGxpbmciLAogICAgImJvb2siLAogICAgImNhbGVuZGFyIiwKICAgICJjaGFyZ2UiLAogICAgImVycm9yIiwKICAgICJmYWlsZWQiLAogICAgImhlbHAiLAogICAgImludml0YXRpb24iLAogICAgImludm9pY2UiLAogICAgImlzc3VlIiwKICAgICJsb2dpbiIsCiAgICAibWVldGluZyIsCiAgICAicGFzc3dvcmQiLAogICAgInBheW1lbnQiLAogICAgInJlY2VpcHQiLAogICAgInJlZnVuZCIsCiAgICAicmVwb3J0IiwKICAgICJyZXF1ZXN0IiwKICAgICJyZXNjaGVkdWxlIiwKICAgICJyZXNldCIsCiAgICAic2NoZWR1bGUiLAogICAgInNvZnR3YXJlIiwKICAgICJzdXBwb3J0IiwKICAgICJ0aW1lIiwKICAgICJ0b21vcnJvdyIKICBdLAogICJ3ZWlnaHRzIjogWwogICAgWwogICAgICAtMC4yODA0NjAxMTI3ODM2MTU2NiwKICAgICAgMC45NDY0MDYwNDY4Nzc0NTg5LAogICAgICAtMC40ODI3OTI4NTczNTQxMzQ5LAogICAgICAxLjAzNjY5NjUzNDk5MjYyNTgsCiAgICAgIC0wLjQ4MjQ5NDI0NTUzNDIwNDcsCiAgICAgIC0wLjQ0NjEwNzk2MTI3NDIxOTE0LAogICAgICAwLjk4MTk2Nzc0NzEzMjY4MTIsCiAgICAgIC0wLjUwOTc1MDkxMTQyODE3MDcsCiAgICAgIDAuNDIwMzY3Njc4NjkxODE3MiwKICAgICAgLTAuNDYyMzY3NTcxOTQ2ODM0OSwKICAgICAgLTAuNDQwMDc3MTEwODc0NTE0LAogICAgICAxLjMzMDczMDA3NjQxOTQzOTQsCiAgICAgIC0wLjQ2MzcwNTE5NzMxNjc3MzQ1LAogICAgICAtMC40NDc4NzgzNDk3NzE5MTUyNiwKICAgICAgLTAuNjUwOTA2NjM3NTIwMzQ5LAogICAgICAtMC41MTMyMDIxMTU5MTk3NDgsCiAgICAgIDEuMjg3NjY4NDUwNTM4NjUyNCwKICAgICAgMC44NjQ0NTYxOTk0MTY3MDI3LAogICAgICAwLjkxMDg0MDEwOTYwMTkzNDYsCiAgICAgIC0wLjI4MTY1NTY1ODExMDQ3MDMzLAogICAgICAwLjU1NTcxMDg2ODAwNjkyNDIsCiAgICAgIC0wLjI1Nzc0MDEwNTk5NjI1ODYzLAogICAgICAtMC41MDA0OTc1MjYwMzA5MTQ2LAogICAgICAtMC40NDEzMDg1Mzg2MTA1NTU3LAogICAgICAtMC41MjQ3OTQ3MTMyNTkwNzgxLAogICAgICAtMC40Njk4Mzk3MzYzMzgwNTYzLAogICAgICAtMC40MzgzMzE0MjY2NjA1NTcxMywKICAgICAgLTAuNDgwNTgxMDc2Mjk3NzQzNgogICAgXSwKICAgIFsKICAgICAgMC41MzkxMTU4NTU2MDMwMDMzLAogICAgICAtMC40NzYzMzIyNTk2ODY5NTUyLAogICAgICAtMC41MDgxODUzMjA2ODk1MjM1LAogICAgICAtMC41MjA2NDQxMTE3OTU4MzE2LAogICAgICAtMC41MDk3ODcyOTM4MTEzNDMsCiAgICAgIC0wLjQ5MzEzMDk2NTI1OTkyODY3LAogICAgICAtMC41MDk3NTAwNTMxNzIzNDIyLAogICAgICAxLjAxMzg1OTAzNzQzMDY0NDIsCiAgICAgIC0wLjIxNzExNDU1MTM1MjgzODY0LAogICAgICAwLjkyMDQ1MzMyNDIxODMwODksCiAgICAgIC0wLjQ2NTc1NDE5MDcwNTAyODYsCiAgICAgIC0wLjY4MzA5Mjk0NDEwNjA4MTMsCiAgICAgIDAuOTI4ODU3NDUwNDUxOTYwMiwKICAgICAgMC45MDIxNzg3Mjc4NDI4MTgsCiAgICAgIC0wLjY3Nzg3NDY5NjAwNzYzODYsCiAgICAgIDEuMDE0NDI5MTM5MTA1MDM2OCwKICAgICAgLTAuNjY2MDM1NzMwNDI5MjM1NSwKICAgICAgLTAuNDM4NzA2NjM1NjIyMDMwMTYsCiAgICAgIC0wLjQ2MzU3NzcwMDQ4MzUzMDUsCiAgICAgIDAuNTcwMjg5MDU3NDA2NzI2MiwKICAgICAgLTAuMjcwOTQ3OTc2NTA4MTc1MzcsCiAgICAgIC0wLjI2NDIwOTI0NzIwMDk3MTksCiAgICAgIDEuMDIwMDUyNzQ3MzgwMjk3MiwKICAgICAgLTAuNDU4MDA5ODEyODgyODY2MSwKICAgICAgMS4wNDE3MTI2OTQ2Mjg5NjIzLAogICAgICAwLjkzNDkzNjU4MTUxNTg0OTcsCiAgICAgIC0wLjQ1NzExMDAzMDgxMjI0MjQsCiAgICAgIC0wLjUwNDYwNjQzMzY3MTEyNAogICAgXSwKICAgIFsKICAgICAgLTAuMjc3NzgwNDE0NzgzMjk2OSwKICAgICAgLTAuNDcyMDYwNDgzMzA4MDc2NSwKICAgICAgMS4wMDIwNjY1NTQyMDcyMzU4LAogICAgICAtMC41MjU2OTUyMjc4MDMzODExLAogICAgICAxLjAwMTQ2MzYyNDk4MjQ5MDksCiAgICAgIDAuOTI0MDgzNDM0NjI4ODI4LAogICAgICAtMC40ODU5NTk3ODQ3NDAwNDcyLAogICAgICAtMC41MDc5NTI4Mzk1NDAxOTc4LAogICAgICAtMC4yMDcyOTczMTA2MjA4NTM0NiwKICAgICAgLTAuNDYxNzM2ODg1ODU4ODA5NzMsCiAgICAgIDAuOTAwMzEwNzA5MDY4ODIxOCwKICAgICAgLTAuNjQ1MjE4MDEyODc0MTY5OCwKICAgICAgLTAuNDY4MTE2MjE1MDc2NTI4MSwKICAgICAgLTAuNDQ4NDg0OTUxNTMwNjk0OSwKICAgICAgMS4zMTgzNTgzNTA5MzMzNzQ1LAogICAgICAtMC40OTg3NTI1MTQ3OTI0Njg3NCwKICAgICAgLTAuNjIxMjU5NTQ4MzU2MzA4OCwKICAgICAgLTAuNDA1NDQ0OTkxOTUxMDI2MywKICAgICAgLTAuNDM4Nzc3OTI0MzM3NjMwOSwKICAgICAgLTAuMjg3NTA1MTI5MjQ5ODg4NzUsCiAgICAgIC0wLjI3MjkzMzA1NDc0OTE3MTksCiAgICAgIDAuNTEyMDEyODE2NDI1MDczLAogICAgICAtMC41MTIzMzEwOTY2NTc0NDU1LAogICAgICAwLjg4NjcwNTc1ODUxMTE1LAogICAgICAtMC41MzIzMDk0NTU2ODg4OTk5LAogICAgICAtMC40NzUyNTYyNjU5MzUyODY0NywKICAgICAgMC44NzU4Mjk4NzMwMzQ2NzQ0LAogICAgICAwLjk5NjY1Mzk1ODkzMDIzNTEKICAgIF0KICBdLAogICJiaWFzIjogWwogICAgLTAuMDU5ODkzODkzMjU0MTA3MzIsCiAgICAwLjEwNjY0MTkwODY4MjkxNzQ1LAogICAgLTAuMDQ2NzQ4MDE1NDI4ODEwMzQ1CiAgXQp9Cg==")
assert hashlib.sha256(REFERENCE_MODEL_BYTES).hexdigest() == "0cc31078081eb0d4a349d55b1f748451c7e3f4da0b5a0c9ea75f5b8561e5cc77"
REFERENCE_MODEL = json.loads(REFERENCE_MODEL_BYTES)


In [ ]:
# This source is embedded after the trusted model implementation and fixed fixtures.
import os
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

def run_notebook(job,data,reference_model):
    destination=Path(os.environ.get('TRAINER_OUTPUT_DIR','/kaggle/working'))
    destination.mkdir(parents=True,exist_ok=True)
    events=[];started=time.perf_counter();stamp=datetime.now(timezone.utc).isoformat()
    def save(name,value):
        path=destination/name
        if path.is_symlink():raise ValueError('Output symlinks are not allowed')
        path.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False))
    def event(row):
        events.append(row);print(json.dumps(row,allow_nan=False))
    def peak_memory():
        try:
            import resource
            value=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            return int(value/1024 if sys.platform=='darwin' else value)
        except (ImportError,AttributeError):return None
    try:
        if sys.version_info<(3,10):raise RuntimeError('Python 3.10 or newer required')
        validate_dataset(data);event({'event':'dataset_loaded','train':18,'validation':6,'test':6})
        if job['experiment_type']=='inference':
            model=validate_model(reference_model);predictions=[];total_tokens=0;latency=0.0
            for text in job['config']['inputs']:
                begin=time.perf_counter();p=predict(model,text);elapsed=(time.perf_counter()-begin)*1000
                predictions.append({'input':text,'tokens':tokenize(text),**p,'latency_ms':elapsed});total_tokens+=p['word_tokens'];latency+=elapsed
            metrics={'prediction_count':len(predictions),'input_word_tokens':total_tokens,'latency_ms_total':latency,'peak_process_memory_kib':peak_memory()}
            details={'predictions':predictions,'scores_are_calibrated':False,'tokenizer':TOKENIZER}
            event({'event':'inference_complete','examples':len(predictions)})
        elif job['experiment_type']=='training':
            result=train(data,job['config'],event)
            metrics={'before_validation_accuracy':result['before_validation']['accuracy'],'after_validation_accuracy':result['after_validation']['accuracy'],'heldout_test_accuracy':result['heldout_test']['accuracy'],
              'training_loss_last':result['history'][-1]['training_loss'],'validation_loss_best':result['after_validation']['loss'],'epochs_completed':len(result['history']),'selected_epoch':result['selected_epoch'],'peak_process_memory_kib':peak_memory()}
            details={'before_validation':result['before_validation'],'after_validation':result['after_validation'],'heldout_test':result['heldout_test'],'selected_epoch':result['selected_epoch']}
            save('checkpoint-best.json',result['model']);save('history.json',result['history']);save('splits.json',result['splits'])
        else:raise ValueError('Unsupported fixed experiment recipe')
        report={'schema_version':1,'status':'completed','spec_sha256':digest(job),'recipe_id':job['recipe_id'],'experiment_type':job['experiment_type'],
          'provenance':{'source':'notebook-process','python':platform.python_version(),'libraries':{'stdlib':platform.python_version()},'platform':platform.system(),'started_at':stamp,'duration_ms':(time.perf_counter()-started)*1000,'model_kind':'linear-softmax','parameters':87,'dataset':'synthetic-intents-30-v1','remote_verified':False},
          'metrics':metrics,'details':details,'errors':[]}
        save('metrics.json',metrics);save('result.json',report);save('events.json',events)
        event({'event':'artifacts_written','count':6 if job['experiment_type']=='training' else 3})
        return report
    except Exception as error:
        # Do not log inputs, environment variables or arbitrary exception values.
        failed={'schema_version':1,'status':'failed','spec_sha256':digest(job),'recipe_id':job['recipe_id'],'errors':[{'code':'NOTEBOOK_FAILURE','type':type(error).__name__}],'metrics':{}}
        save('result.json',failed);event({'event':'failed','error_type':type(error).__name__});raise

REPORT = run_notebook(JOB, DATA, REFERENCE_MODEL)
